# Ridge and Lasso Regression, Explained Simply
Regularisation: stopping a model from overfitting by penalising big weights. Runs free in Google Colab.

Run each cell from top to bottom with **Shift + Enter**.

## 1. The data
A rental website has 80 apartments with 14 facts each. Some facts matter (size, distance), some are pure noise (street number, letterbox). We learn from only 20 apartments, which makes overfitting easy.

### 80 apartments, 14 facts each
- `apartments.csv`: 80 apartments: size, bedrooms, distance to the city, floor, age, balcony, parking, plus 7 facts that shouldn't matter, and the monthly rent
- `list(df.columns)`: the column names

In [ ]:
import pandas as pd

url = ("https://raw.githubusercontent.com/Uttam-Nepal/"
       "machine-learning-explained/main/"
       "ridge-and-lasso/apartments.csv")
df = pd.read_csv(url)
print(df.shape)
print(list(df.columns))

### Learn from just 20 apartments
- `df.drop(columns="rent")`: every column except the rent: 14 inputs
- `train_size=20`: only 20 apartments to learn from; 60 kept for testing

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns="rent")
y = df["rent"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=20, random_state=0)
print(X_train.shape, X_test.shape)

## 2. Plain linear regression overfits
14 inputs and only 20 examples: plenty of freedom to memorise.

### Linear regression
- `StandardScaler()`: scale every input first; the penalties later need this
- `score`: R squared: 1 is perfect, 0 is no better than guessing the average
- `the output`: almost perfect on its 20 apartments, poor on new ones: overfitting

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

plain = make_pipeline(StandardScaler(), LinearRegression())
plain.fit(X_train, y_train)
print("train R squared:", round(plain.score(X_train, y_train), 3))
print("test R squared: ", round(plain.score(X_test, y_test), 3))

## 3. Ridge and Lasso
**Ridge**: loss = mean squared error + alpha x (sum of squared weights). **Lasso**: loss = mean squared error + alpha x (sum of absolute weights). **alpha** sets how strong the penalty is.

### Ridge and Lasso
- `Ridge(alpha=3)`: linear regression with the squared-weights penalty
- `Lasso(alpha=15)`: linear regression with the absolute-weights penalty
- `the output`: slightly worse on training, much better on new apartments

In [ ]:
from sklearn.linear_model import Ridge, Lasso

ridge = make_pipeline(StandardScaler(), Ridge(alpha=3))
ridge.fit(X_train, y_train)
lasso = make_pipeline(StandardScaler(), Lasso(alpha=15))
lasso.fit(X_train, y_train)

for name, m in [("plain", plain), ("ridge", ridge), ("lasso", lasso)]:
    tr, te = m.score(X_train, y_train), m.score(X_test, y_test)
    print(f"{name:>5}: train {tr:.3f}   test {te:.3f}")

### Compare the weights
- `m[-1]`: the last step of each pipeline: the model itself
- `.coef_`: its weights, one per input
- `ridge`: every weight shrunk towards zero
- `lasso`: the useless inputs set to exactly zero

In [ ]:
models = {"plain": plain, "ridge": ridge, "lasso": lasso}
weights = pd.DataFrame({name: m[-1].coef_ for name, m in models.items()},
                       index=X.columns).round(0) + 0.0
print(weights)

### The weights as a chart
- `weights.plot.barh`: a horizontal bar for every weight, for all three models
- `the picture`: plain: big weights on useless inputs; Lasso: zeros

In [ ]:
import matplotlib.pyplot as plt

weights.plot.barh(figsize=(8, 6))
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("weight (on scaled inputs)")
plt.title("Plain vs Ridge vs Lasso")
plt.show()

### Lasso picks the useful inputs
- `weights["lasso"] != 0`: True for inputs Lasso kept
- `dropped`: the inputs Lasso decided are useless

In [ ]:
kept = weights.index[weights["lasso"] != 0]
dropped = weights.index[weights["lasso"] == 0]
print("kept:   ", list(kept))
print("dropped:", list(dropped))

## 4. Choosing alpha
Too small: overfitting. Too big: underfitting. Cross-validation tries many alphas on different splits of the training data and picks the best.

### Test score for many alphas
- `np.logspace(-2, 3, 30)`: 30 alphas from 0.01 to 1,000, evenly spread on a log scale
- `plt.semilogx`: a chart with a log scale across
- `the shape`: low on the left (overfitting), best in the middle, crashing on the right (underfitting)

*We peek at the test set here only to draw the picture. To actually choose alpha, use cross-validation on the training data, as in the next cell.*

In [ ]:
import numpy as np

alphas = np.logspace(-2, 3, 30)
scores = [make_pipeline(StandardScaler(), Lasso(alpha=a, max_iter=10000))
          .fit(X_train, y_train).score(X_test, y_test) for a in alphas]
plt.semilogx(alphas, scores, marker="o")
plt.xlabel("alpha (penalty strength)"); plt.ylabel("test R squared")
plt.title("Lasso: too weak, just right, too strong"); plt.show()

### Let cross-validation choose
- `RepeatedKFold`: split the training data 5 ways, and repeat that 10 times with different shuffles: with only 20 apartments, one split is too noisy
- `LassoCV(cv=splits)`: try many alphas on all those splits, keep the best on average
- `alpha_`: the alpha it chose, without ever touching the test set

In [ ]:
from sklearn.linear_model import LassoCV
from sklearn.model_selection import RepeatedKFold

splits = RepeatedKFold(n_splits=5, n_repeats=10, random_state=0)
cv = make_pipeline(StandardScaler(), LassoCV(cv=splits, random_state=0))
cv.fit(X_train, y_train)
print("chosen alpha:", round(cv[-1].alpha_, 1))
print("test R squared:", round(cv.score(X_test, y_test), 3))

## Try it yourself
1. Try Lasso with alpha = 1 and alpha = 100. How many weights are zero each time?
2. Use 60 training apartments instead of 20 (train_size=60). Does regularisation still help as much?
3. Try ElasticNet(alpha=5, l1_ratio=0.5) in the same pipeline.